# 4. 马尔可夫链蒙特卡洛采样

本节介绍马尔可夫链蒙特卡洛（Markov Chain Monte Carlo, MCMC）采样方法，这是变分蒙特卡洛方法中用于从神经网络量子态分布中采样的关键技术。

## 4.1 马尔可夫链基础

马尔可夫链是一种随机过程，其特点是未来的状态只依赖于当前状态，与过去的状态无关。在变分蒙特卡洛方法中，我们使用马尔可夫链来从波函数的概率分布$|\psi(\mathbf{x})|^2$中采样。

### 马尔可夫链的定义

马尔可夫链是一组随机变量$\{X_0, X_1, X_2, \ldots\}$，满足马尔可夫性质：

$$P(X_{t+1} = x_{t+1} | X_t = x_t, X_{t-1} = x_{t-1}, \ldots, X_0 = x_0) = P(X_{t+1} = x_{t+1} | X_t = x_t)$$

### 转移概率

转移概率$T(\mathbf{x} \to \mathbf{x}')$表示从状态$\mathbf{x}$转移到状态$\mathbf{x}'$的概率。转移概率矩阵满足：

$$\sum_{\mathbf{x}'} T(\mathbf{x} \to \mathbf{x}') = 1$$

### 平稳分布

如果存在一个概率分布$\pi(\mathbf{x})$，使得：

$$\pi(\mathbf{x}') = \sum_{\mathbf{x}} \pi(\mathbf{x}) T(\mathbf{x} \to \mathbf{x}')$$

则称$\pi(\mathbf{x})$是马尔可夫链的平稳分布。在变分蒙特卡洛方法中，我们希望构造一个马尔可夫链，其平稳分布为$|\psi(\mathbf{x})|^2$。

### 详细平衡条件

如果转移概率满足详细平衡条件：

$$\pi(\mathbf{x}) T(\mathbf{x} \to \mathbf{x}') = \pi(\mathbf{x}') T(\mathbf{x}' \to \mathbf{x})$$

则$\pi(\mathbf{x})$是马尔可夫链的平稳分布。详细平衡条件是构造马尔可夫链的重要准则。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set style for plots
plt.style.use('seaborn-v0_8')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define a simple RBM wave function for demonstration
class SimpleRBM(nn.Module):
    def __init__(self, n_visible, n_hidden):
        super(SimpleRBM, self).__init__()
        self.n_visible = n_visible
        self.n_hidden = n_hidden
        
        # Parameters
        self.a = nn.Parameter(torch.randn(n_visible))  # Visible bias
        self.b = nn.Parameter(torch.randn(n_hidden))  # Hidden bias
        self.W = nn.Parameter(torch.randn(n_visible, n_hidden))  # Weights
        
    def forward(self, x):
        # Visible layer contribution
        visible_term = torch.sum(x * self.a, dim=1)
        
        # Hidden layer contribution
        hidden_term = torch.sum(
            torch.log(2 * torch.cosh(torch.matmul(x, self.W) + self.b)),
            dim=1
        )
        
        return visible_term + hidden_term
    
    def log_psi(self, x):
        return self.forward(x)
    
    def psi(self, x):
        return torch.exp(self.log_psi(x))
    
    def probability(self, x):
        """
        Compute probability |psi(x)|^2
        """
        return torch.exp(2 * self.log_psi(x))

# Create a simple RBM for demonstration
n_spins = 4
n_hidden = 4
rbm = SimpleRBM(n_spins, n_hidden).to(device)

# Generate all possible spin configurations for a small system
def generate_all_spin_configurations(n_spins):
    """
    Generate all possible spin configurations for a system of n_spins
    
    Returns:
    configurations: array of shape (2^n_spins, n_spins)
    """
    n_configs = 2 ** n_spins
    configurations = torch.zeros(n_configs, n_spins, device=device)
    
    for i in range(n_configs):
        # Convert integer to binary representation
        binary_str = bin(i)[2:].zfill(n_spins)
        # Convert to +/- 1
        configurations[i] = torch.tensor([1 if bit == '1' else -1 for bit in binary_str], device=device)
    
    return configurations

# Generate all configurations
all_configs = generate_all_spin_configurations(n_spins)

# Compute probabilities for all configurations
probs = rbm.probability(all_configs)
probs = probs / torch.sum(probs)  # Normalize

print(f"Total number of configurations: {len(all_configs)}")
print(f"Shape of configurations: {all_configs.shape}")
print(f"Shape of probabilities: {probs.shape}")
print(f"Sum of probabilities: {torch.sum(probs).item()}")

# Plot probabilities
plt.figure(figsize=(12, 6))
plt.bar(range(len(probs)), probs.cpu().detach().numpy())
plt.xlabel('Configuration Index')
plt.ylabel('Probability')
plt.title('Probability Distribution of Spin Configurations')
plt.grid(True)
plt.show()

## 4.2 Metropolis-Hastings算法

Metropolis-Hastings算法是最常用的MCMC采样方法之一，它通过构造满足详细平衡条件的马尔可夫链来从目标分布中采样。

### 算法步骤

Metropolis-Hastings算法的步骤如下：

1. **初始化**：选择初始状态$\mathbf{x}_0$

2. **提议**：从提议分布$q(\mathbf{x}' | \mathbf{x}_t)$中生成一个新状态$\mathbf{x}'$

3. **接受概率**：计算接受概率：

$$A(\mathbf{x}_t \to \mathbf{x}') = \min\left(1, \frac{\pi(\mathbf{x}') q(\mathbf{x}_t | \mathbf{x}')}{\pi(\mathbf{x}_t) q(\mathbf{x}' | \mathbf{x}_t)}\right)$$

4. **接受或拒绝**：生成一个均匀随机数$u \in [0, 1]$，如果$u < A(\mathbf{x}_t \to \mathbf{x}')$，则接受新状态，$\mathbf{x}_{t+1} = \mathbf{x}'$；否则，保持原状态，$\mathbf{x}_{t+1} = \mathbf{x}_t$

5. **重复**：重复步骤2-4，直到获得足够的样本

### 对称提议分布

如果提议分布是对称的，即$q(\mathbf{x}' | \mathbf{x}_t) = q(\mathbf{x}_t | \mathbf{x}')$，则接受概率简化为：

$$A(\mathbf{x}_t \to \mathbf{x}') = \min\left(1, \frac{\pi(\mathbf{x}')}{\pi(\mathbf{x}_t)}\right)$$

这被称为Metropolis算法。

### 自旋翻转提议

在自旋系统中，常用的提议分布是随机翻转一个自旋：

$$q(\mathbf{x}' | \mathbf{x}) = \frac{1}{N} \sum_{i=1}^{N} \delta(\mathbf{x}', \mathbf{x}^{(i)})$$

其中$\mathbf{x}^{(i)}$是翻转第$i$个自旋后的组态，$N$是自旋总数。这种提议分布是对称的，因此可以使用Metropolis算法。

In [ ]:
def metropolis_hastings_step(wavefunction, current_state):
    """
    Perform one step of Metropolis-Hastings algorithm
    
    Parameters:
    wavefunction: neural network wave function
    current_state: current spin configuration, shape (n_visible,)
    
    Returns:
    new_state: new spin configuration, shape (n_visible,)
    accepted: whether the move was accepted
    """
    n_spins = current_state.size(0)
    
    # Propose a new state by flipping a random spin
    flip_idx = torch.randint(0, n_spins, (1,)).item()
    proposed_state = current_state.clone()
    proposed_state[flip_idx] *= -1
    
    # Compute probabilities (|psi|^2)
    current_prob = wavefunction.probability(current_state.unsqueeze(0))
    proposed_prob = wavefunction.probability(proposed_state.unsqueeze(0))
    
    # Compute acceptance probability
    acceptance_prob = proposed_prob / current_prob
    acceptance_prob = min(1.0, acceptance_prob.item())
    
    # Accept or reject
    if torch.rand(1).item() < acceptance_prob:
        return proposed_state, True
    else:
        return current_state, False

def metropolis_hastings_sampling(wavefunction, initial_state, n_steps, burn_in=0):
    """
    Perform Metropolis-Hastings sampling
    
    Parameters:
    wavefunction: neural network wave function
    initial_state: initial spin configuration, shape (n_visible,)
    n_steps: number of MCMC steps
    burn_in: number of burn-in steps (discarded)
    
    Returns:
    samples: array of samples, shape (n_steps - burn_in, n_visible)
    acceptance_rate: acceptance rate
    """
    current_state = initial_state.clone()
    samples = []
    n_accepted = 0
    
    for step in range(n_steps):
        # Perform one MCMC step
        current_state, accepted = metropolis_hastings_step(wavefunction, current_state)
        
        # Count accepted moves
        if accepted:
            n_accepted += 1
        
        # Store sample after burn-in
        if step >= burn_in:
            samples.append(current_state.clone())
    
    # Convert to tensor
    samples = torch.stack(samples, dim=0)
    
    # Compute acceptance rate
    acceptance_rate = n_accepted / n_steps
    
    return samples, acceptance_rate

# Example: Perform Metropolis-Hastings sampling
n_spins = 4
n_hidden = 4
rbm = SimpleRBM(n_spins, n_hidden).to(device)

# Sampling parameters
n_steps = 10000
burn_in = 1000

# Initial state (all spins up)
initial_state = torch.ones(n_spins, device=device)

# Perform sampling
samples, acceptance_rate = metropolis_hastings_sampling(rbm, initial_state, n_steps, burn_in)

print(f"Number of samples: {len(samples)}")
print(f"Acceptance rate: {acceptance_rate:.4f}")
print(f"Shape of samples: {samples.shape}")

# Compute empirical probabilities
unique_configs, counts = torch.unique(samples, dim=0, return_counts=True)
empirical_probs = counts.float() / torch.sum(counts)

# Compute exact probabilities for comparison
all_configs = generate_all_spin_configurations(n_spins)
exact_probs = rbm.probability(all_configs)
exact_probs = exact_probs / torch.sum(exact_probs)

# Create a mapping from configuration to probability
config_to_prob = {}
for config, prob in zip(all_configs, exact_probs):
    config_tuple = tuple(config.tolist())
    config_to_prob[config_tuple] = prob.item()

# Get exact probabilities for sampled configurations
exact_probs_sampled = torch.tensor([config_to_prob[tuple(config.tolist())] for config in unique_configs], device=device)

# Plot comparison
plt.figure(figsize=(12, 6))
x = np.arange(len(unique_configs))
width = 0.35

plt.bar(x - width/2, empirical_probs.cpu().detach().numpy(), width, label='Empirical')
plt.bar(x + width/2, exact_probs_sampled.cpu().detach().numpy(), width, label='Exact')

plt.xlabel('Configuration Index')
plt.ylabel('Probability')
plt.title('Empirical vs Exact Probabilities')
plt.legend()
plt.grid(True)
plt.show()

## 4.3 退火方法

退火方法（Annealing）是一种改进MCMC采样效率的技术，通过在采样过程中逐渐调整系统的"温度"，帮助马尔可夫链探索状态空间并避免陷入局部极小值。

### 模拟退火

模拟退火（Simulated Annealing）借鉴了固体退火过程的物理原理，通过在高温下允许系统探索更大的状态空间，然后逐渐降低温度，使系统趋于稳定状态。

在模拟退火中，我们修改接受概率为：

$$A(\mathbf{x}_t \to \mathbf{x}') = \min\left(1, \left(\frac{\pi(\mathbf{x}')}{\pi(\mathbf{x}_t)}\right)^{1/T}\right)$$

其中$T$是温度参数。当$T$较高时，接受概率接近于1，系统可以自由探索状态空间；当$T$较低时，接受概率接近于标准Metropolis算法。

### 退火调度

退火调度（Annealing Schedule）是温度随时间变化的策略。常见的退火调度包括：

1. **指数退火**：$T(t) = T_0 \alpha^t$，其中$\alpha < 1$
2. **线性退火**：$T(t) = T_0 - \beta t$，其中$\beta > 0$
3. **对数退火**：$T(t) = T_0 / \log(t + 1)$

### 并行退火

并行退火（Parallel Tempering）是一种更高级的退火技术，它同时运行多个不同温度的马尔可夫链，并允许链之间交换状态。这种方法可以更有效地探索状态空间，特别是在复杂的多峰分布中。

在并行退火中，我们有$K$个温度$T_1 < T_2 < \ldots < T_K$，每个温度对应一个马尔可夫链。除了在每个链内执行标准的Metropolis步骤外，我们还定期尝试交换相邻链的状态。

In [ ]:
def metropolis_hastings_step_annealed(wavefunction, current_state, temperature):
    """
    Perform one step of Metropolis-Hastings algorithm with annealing
    
    Parameters:
    wavefunction: neural network wave function
    current_state: current spin configuration, shape (n_visible,)
    temperature: temperature parameter
    
    Returns:
    new_state: new spin configuration, shape (n_visible,)
    accepted: whether the move was accepted
    """
    n_spins = current_state.size(0)
    
    # Propose a new state by flipping a random spin
    flip_idx = torch.randint(0, n_spins, (1,)).item()
    proposed_state = current_state.clone()
    proposed_state[flip_idx] *= -1
    
    # Compute log probabilities (2 * log|psi|)
    current_log_prob = 2 * wavefunction.log_psi(current_state.unsqueeze(0))
    proposed_log_prob = 2 * wavefunction.log_psi(proposed_state.unsqueeze(0))
    
    # Compute log acceptance probability with annealing
    log_acceptance_prob = (proposed_log_prob - current_log_prob) / temperature
    acceptance_prob = min(1.0, torch.exp(log_acceptance_prob).item())
    
    # Accept or reject
    if torch.rand(1).item() < acceptance_prob:
        return proposed_state, True
    else:
        return current_state, False

def simulated_annealing_sampling(wavefunction, initial_state, n_steps, burn_in, initial_temp, final_temp):
    """
    Perform simulated annealing sampling
    
    Parameters:
    wavefunction: neural network wave function
    initial_state: initial spin configuration, shape (n_visible,)
    n_steps: number of MCMC steps
    burn_in: number of burn-in steps (discarded)
    initial_temp: initial temperature
    final_temp: final temperature
    
    Returns:
    samples: array of samples, shape (n_steps - burn_in, n_visible)
    acceptance_rate: acceptance rate
    temperatures: array of temperatures used
    """
    current_state = initial_state.clone()
    samples = []
    n_accepted = 0
    temperatures = []
    
    # Exponential annealing schedule
    alpha = (final_temp / initial_temp) ** (1 / n_steps)
    temperature = initial_temp
    
    for step in range(n_steps):
        # Perform one MCMC step with annealing
        current_state, accepted = metropolis_hastings_step_annealed(
            wavefunction, current_state, temperature
        )
        
        # Count accepted moves
        if accepted:
            n_accepted += 1
        
        # Store sample after burn-in
        if step >= burn_in:
            samples.append(current_state.clone())
            temperatures.append(temperature)
        
        # Update temperature
        temperature *= alpha
    
    # Convert to tensor
    samples = torch.stack(samples, dim=0)
    temperatures = torch.tensor(temperatures)
    
    # Compute acceptance rate
    acceptance_rate = n_accepted / n_steps
    
    return samples, acceptance_rate, temperatures

# Example: Perform simulated annealing sampling
n_spins = 4
n_hidden = 4
rbm = SimpleRBM(n_spins, n_hidden).to(device)

# Sampling parameters
n_steps = 10000
burn_in = 1000
initial_temp = 5.0
final_temp = 0.1

# Initial state (all spins up)
initial_state = torch.ones(n_spins, device=device)

# Perform sampling
samples_sa, acceptance_rate_sa, temperatures = simulated_annealing_sampling(
    rbm, initial_state, n_steps, burn_in, initial_temp, final_temp
)

print(f"Number of samples: {len(samples_sa)}")
print(f"Acceptance rate: {acceptance_rate_sa:.4f}")
print(f"Shape of samples: {samples_sa.shape}")

# Plot temperature schedule
plt.figure(figsize=(12, 6))
plt.plot(temperatures.cpu().numpy())
plt.xlabel('MCMC Step (after burn-in)')
plt.ylabel('Temperature')
plt.title('Temperature Schedule in Simulated Annealing')
plt.grid(True)
plt.show()

# Compare standard MCMC and simulated annealing
samples_std, acceptance_rate_std = metropolis_hastings_sampling(
    rbm, initial_state, n_steps, burn_in
)

# Compute empirical probabilities
unique_configs_std, counts_std = torch.unique(samples_std, dim=0, return_counts=True)
empirical_probs_std = counts_std.float() / torch.sum(counts_std)

unique_configs_sa, counts_sa = torch.unique(samples_sa, dim=0, return_counts=True)
empirical_probs_sa = counts_sa.float() / torch.sum(counts_sa)

# Compute exact probabilities for comparison
all_configs = generate_all_spin_configurations(n_spins)
exact_probs = rbm.probability(all_configs)
exact_probs = exact_probs / torch.sum(exact_probs)

# Create a mapping from configuration to probability
config_to_prob = {}
for config, prob in zip(all_configs, exact_probs):
    config_tuple = tuple(config.tolist())
    config_to_prob[config_tuple] = prob.item()

# Get exact probabilities for sampled configurations
exact_probs_std = torch.tensor([config_to_prob[tuple(config.tolist())] for config in unique_configs_std], device=device)
exact_probs_sa = torch.tensor([config_to_prob[tuple(config.tolist())] for config in unique_configs_sa], device=device)

# Plot comparison
plt.figure(figsize=(12, 6))
x = np.arange(len(all_configs))
width = 0.25

plt.bar(x - width, exact_probs.cpu().detach().numpy(), width, label='Exact')

# For standard MCMC
indices_std = [torch.all(all_configs == config, dim=1).nonzero(as_tuple=True)[0].item() 
              for config in unique_configs_std]
plt.bar(indices_std, empirical_probs_std.cpu().detach().numpy(), width, label='Standard MCMC')

# For simulated annealing
indices_sa = [torch.all(all_configs == config, dim=1).nonzero(as_tuple=True)[0].item() 
             for config in unique_configs_sa]
plt.bar(indices_sa + width, empirical_probs_sa.cpu().detach().numpy(), width, label='Simulated Annealing')

plt.xlabel('Configuration Index')
plt.ylabel('Probability')
plt.title('Comparison of Sampling Methods')
plt.legend()
plt.grid(True)
plt.show()

## 4.4 采样效率评估

在变分蒙特卡洛方法中，采样效率直接影响能量计算的准确性和优化过程的稳定性。因此，评估采样效率是非常重要的。

### 自相关时间

自相关时间（Autocorrelation Time）是衡量马尔可夫链混合效率的重要指标。它表示连续样本之间的相关性程度，自相关时间越短，采样效率越高。

自相关函数定义为：

$$A(\tau) = \frac{\langle O(t) O(t+\tau) \rangle - \langle O \rangle^2}{\langle O^2 \rangle - \langle O \rangle^2}$$

其中$O(t)$是观测量的时间序列，$\tau$是时间延迟。

积分自相关时间定义为：

$$\tau_{int} = \frac{1}{2} + \sum_{\tau=1}^{\infty} A(\tau)$$

### 有效样本大小

有效样本大小（Effective Sample Size, ESS）表示独立样本的等效数量，考虑了样本之间的相关性。ESS越大，采样效率越高。

ESS的计算公式为：

$$N_{eff} = \frac{N}{1 + 2 \sum_{\tau=1}^{M} A(\tau)}$$

其中$N$是总样本数，$M$是截断时间，$A(\tau)$是自相关函数。

### 接受率

接受率（Acceptance Rate）是Metropolis-Hastings算法中被接受提议的比例。接受率过高或过低都表明采样效率不佳。

对于自旋翻转提议，最优接受率通常在40%-60%之间。接受率过低表明提议步长过大，导致大多数提议被拒绝；接受率过高表明提议步长过小，导致状态空间探索不充分。

In [ ]:
def compute_autocorrelation(series, max_lag=None):
    """
    Compute autocorrelation function for a time series
    
    Parameters:
    series: time series data, shape (n_samples,)
    max_lag: maximum lag to compute (default: n_samples // 10)
    
    Returns:
    autocorr: autocorrelation function, shape (max_lag+1,)
    """
    n = len(series)
    if max_lag is None:
        max_lag = n // 10
    
    # Normalize the series
    series = (series - torch.mean(series)) / torch.std(series)
    
    # Compute autocorrelation
    autocorr = torch.zeros(max_lag + 1, device=series.device)
    for lag in range(max_lag + 1):
        if lag == 0:
            autocorr[lag] = 1.0
        else:
            autocorr[lag] = torch.mean(series[:-lag] * series[lag:])
    
    return autocorr

def compute_integrated_autocorrelation_time(autocorr):
    """
    Compute integrated autocorrelation time
    
    Parameters:
    autocorr: autocorrelation function, shape (max_lag+1,)
    
    Returns:
    tau_int: integrated autocorrelation time
    """
    # Find where autocorrelation drops below zero for the first time
    # This is a simple cutoff criterion
    cutoff = 0
    for i in range(1, len(autocorr)):
        if autocorr[i] < 0:
            cutoff = i
            break
    
    if cutoff == 0:
        cutoff = len(autocorr) - 1
    
    # Compute integrated autocorrelation time
    tau_int = 0.5 + torch.sum(autocorr[1:cutoff+1])
    
    return tau_int

def compute_effective_sample_size(series, autocorr=None):
    """
    Compute effective sample size
    
    Parameters:
    series: time series data, shape (n_samples,)
    autocorr: precomputed autocorrelation function (optional)
    
    Returns:
    n_eff: effective sample size
    """
    n = len(series)
    
    if autocorr is None:
        autocorr = compute_autocorrelation(series)
    
    # Compute integrated autocorrelation time
    tau_int = compute_integrated_autocorrelation_time(autocorr)
    
    # Compute effective sample size
    n_eff = n / (1 + 2 * tau_int)
    
    return n_eff

# Example: Evaluate sampling efficiency
n_spins = 4
n_hidden = 4
rbm = SimpleRBM(n_spins, n_hidden).to(device)

# Sampling parameters
n_steps = 20000
burn_in = 2000

# Initial state (all spins up)
initial_state = torch.ones(n_spins, device=device)

# Perform sampling
samples, acceptance_rate = metropolis_hastings_sampling(rbm, initial_state, n_steps, burn_in)

print(f"Number of samples: {len(samples)}")
print(f"Acceptance rate: {acceptance_rate:.4f}")

# Compute magnetization time series
magnetization = torch.mean(samples, dim=1)

# Compute autocorrelation function
max_lag = 1000
autocorr = compute_autocorrelation(magnetization, max_lag)

# Plot autocorrelation function
plt.figure(figsize=(12, 6))
plt.plot(autocorr.cpu().numpy())
plt.xlabel('Lag')
plt.ylabel('Autocorrelation')
plt.title('Autocorrelation Function of Magnetization')
plt.grid(True)
plt.show()

# Compute integrated autocorrelation time
tau_int = compute_integrated_autocorrelation_time(autocorr)
print(f"Integrated autocorrelation time: {tau_int.item():.2f}")

# Compute effective sample size
n_eff = compute_effective_sample_size(magnetization, autocorr)
print(f"Effective sample size: {n_eff.item():.2f}")
print(f"Sampling efficiency: {n_eff.item() / len(samples) * 100:.2f}%")

# Plot magnetization time series
plt.figure(figsize=(12, 6))
plt.plot(magnetization.cpu().numpy())
plt.xlabel('MCMC Step')
plt.ylabel('Magnetization')
plt.title('Magnetization Time Series')
plt.grid(True)
plt.show()

# Compare different proposal distributions
def metropolis_hastings_step_multi_flip(wavefunction, current_state, n_flips=1):
    """
    Perform one step of Metropolis-Hastings algorithm with multiple spin flips
    
    Parameters:
    wavefunction: neural network wave function
    current_state: current spin configuration, shape (n_visible,)
    n_flips: number of spins to flip
    
    Returns:
    new_state: new spin configuration, shape (n_visible,)
    accepted: whether the move was accepted
    """
    n_spins = current_state.size(0)
    
    # Propose a new state by flipping multiple random spins
    flip_indices = torch.randperm(n_spins)[:n_flips]
    proposed_state = current_state.clone()
    proposed_state[flip_indices] *= -1
    
    # Compute probabilities (|psi|^2)
    current_prob = wavefunction.probability(current_state.unsqueeze(0))
    proposed_prob = wavefunction.probability(proposed_state.unsqueeze(0))
    
    # Compute acceptance probability
    acceptance_prob = proposed_prob / current_prob
    acceptance_prob = min(1.0, acceptance_prob.item())
    
    # Accept or reject
    if torch.rand(1).item() < acceptance_prob:
        return proposed_state, True
    else:
        return current_state, False

def metropolis_hastings_sampling_multi_flip(wavefunction, initial_state, n_steps, burn_in, n_flips=1):
    """
    Perform Metropolis-Hastings sampling with multiple spin flips
    
    Parameters:
    wavefunction: neural network wave function
    initial_state: initial spin configuration, shape (n_visible,)
    n_steps: number of MCMC steps
    burn_in: number of burn-in steps (discarded)
    n_flips: number of spins to flip
    
    Returns:
    samples: array of samples, shape (n_steps - burn_in, n_visible)
    acceptance_rate: acceptance rate
    """
    current_state = initial_state.clone()
    samples = []
    n_accepted = 0
    
    for step in range(n_steps):
        # Perform one MCMC step
        current_state, accepted = metropolis_hastings_step_multi_flip(wavefunction, current_state, n_flips)
        
        # Count accepted moves
        if accepted:
            n_accepted += 1
        
        # Store sample after burn-in
        if step >= burn_in:
            samples.append(current_state.clone())
    
    # Convert to tensor
    samples = torch.stack(samples, dim=0)
    
    # Compute acceptance rate
    acceptance_rate = n_accepted / n_steps
    
    return samples, acceptance_rate

# Compare different numbers of spin flips
n_flips_list = [1, 2, 3]
acceptance_rates = []
autocorr_times = []
eff_sample_sizes = []

for n_flips in n_flips_list:
    # Perform sampling
    samples, acceptance_rate = metropolis_hastings_sampling_multi_flip(
        rbm, initial_state, n_steps, burn_in, n_flips
    )
    
    # Compute magnetization time series
    magnetization = torch.mean(samples, dim=1)
    
    # Compute autocorrelation function
    autocorr = compute_autocorrelation(magnetization, max_lag)
    
    # Compute integrated autocorrelation time
    tau_int = compute_integrated_autocorrelation_time(autocorr)
    
    # Compute effective sample size
    n_eff = compute_effective_sample_size(magnetization, autocorr)
    
    # Store results
    acceptance_rates.append(acceptance_rate)
    autocorr_times.append(tau_int.item())
    eff_sample_sizes.append(n_eff.item())
    
    print(f"Number of flips: {n_flips}")
    print(f"Acceptance rate: {acceptance_rate:.4f}")
    print(f"Integrated autocorrelation time: {tau_int.item():.2f}")
    print(f"Effective sample size: {n_eff.item():.2f}")
    print(f"Sampling efficiency: {n_eff.item() / len(samples) * 100:.2f}%")
    print()

# Plot comparison
plt.figure(figsize=(15, 5))

# Acceptance rates
plt.subplot(1, 3, 1)
plt.bar(n_flips_list, acceptance_rates)
plt.xlabel('Number of Spins Flipped')
plt.ylabel('Acceptance Rate')
plt.title('Acceptance Rate vs Number of Flips')
plt.grid(True)

# Autocorrelation times
plt.subplot(1, 3, 2)
plt.bar(n_flips_list, autocorr_times)
plt.xlabel('Number of Spins Flipped')
plt.ylabel('Integrated Autocorrelation Time')
plt.title('Autocorrelation Time vs Number of Flips')
plt.grid(True)

# Effective sample sizes
plt.subplot(1, 3, 3)
plt.bar(n_flips_list, eff_sample_sizes)
plt.xlabel('Number of Spins Flipped')
plt.ylabel('Effective Sample Size')
plt.title('Effective Sample Size vs Number of Flips')
plt.grid(True)

plt.tight_layout()
plt.show()

## 4.5 高级采样技术

除了基本的Metropolis-Hastings算法和退火方法外，还有一些更高级的采样技术，可以进一步提高采样效率，特别是在处理复杂的多峰分布或高维系统时。

### 哈密顿蒙特卡洛

哈密顿蒙特卡洛（Hamiltonian Monte Carlo, HMC）是一种利用物理系统动力学的采样方法。它通过引入辅助动量变量，将采样问题转化为哈密顿系统的模拟，从而能够更有效地探索状态空间。

在HMC中，我们定义哈密顿量：

$$H(\mathbf{x}, \mathbf{p}) = -\log \pi(\mathbf{x}) + \frac{1}{2} \mathbf{p}^T \mathbf{p}$$

其中$\mathbf{x}$是位置变量（系统组态），$\mathbf{p}$是动量变量，$\pi(\mathbf{x})$是目标分布。

HMC的步骤包括：

1. 从标准正态分布中采样动量$\mathbf{p} \sim \mathcal{N}(0, \mathbf{I})$
2. 使用 leapfrog 等方法模拟哈密顿系统的演化
3. 根据Metropolis准则接受或拒绝新状态

### 朗之万动力学

朗之万动力学（Langevin Dynamics）是一种基于随机微分方程的采样方法。它通过添加随机力和摩擦力，使得系统能够更有效地探索状态空间。

朗之万方程为：

$$d\mathbf{x}_t = \nabla \log \pi(\mathbf{x}_t) dt + \sqrt{2} d\mathbf{W}_t$$

其中$\mathbf{W}_t$是维纳过程（布朗运动）。

离散化的朗之万动力学更新规则为：

$$\mathbf{x}_{t+1} = \mathbf{x}_t + \epsilon \nabla \log \pi(\mathbf{x}_t) + \sqrt{2\epsilon} \mathbf{z}_t$$

其中$\epsilon$是步长，$\mathbf{z}_t \sim \mathcal{N}(0, \mathbf{I})$。

### 自适应提议分布

自适应提议分布（Adaptive Proposal Distribution）是一种根据采样历史动态调整提议分布的方法。通过学习目标分布的特征，自适应提议分布可以提高采样效率。

常见的自适应方法包括：

1. **自适应Metropolis**：根据采样历史调整提议分布的协方差矩阵
2. **预条件HMC**：根据目标分布的局部几何结构调整HMC的参数
3. **神经网络提议分布**：使用神经网络学习复杂的提议分布

这些高级采样技术各有特点，适用于不同的场景。在实际应用中，我们需要根据具体问题的特点选择合适的采样方法。

In [ ]:
def langevin_dynamics_step(wavefunction, current_state, step_size):
    """
    Perform one step of Langevin dynamics
    
    Parameters:
    wavefunction: neural network wave function
    current_state: current spin configuration, shape (n_visible,)
    step_size: step size for the update
    
    Returns:
    new_state: new spin configuration, shape (n_visible,)
    """
    # Ensure we're computing gradients
    wavefunction.zero_grad()
    
    # Compute log probability (2 * log|psi|)
    log_prob = 2 * wavefunction.log_psi(current_state.unsqueeze(0))
    
    # Compute gradient
    log_prob.backward()
    
    # Get gradient
    grad = current_state.grad.clone()
    
    # Generate random noise
    noise = torch.randn_like(current_state)
    
    # Update state
    new_state = current_state + step_size * grad + torch.sqrt(2 * step_size) * noise
    
    # Project back to spin values (+/- 1)
    new_state = torch.sign(new_state)
    
    return new_state

def langevin_dynamics_sampling(wavefunction, initial_state, n_steps, burn_in, step_size):
    """
    Perform Langevin dynamics sampling
    
    Parameters:
    wavefunction: neural network wave function
    initial_state: initial spin configuration, shape (n_visible,)
    n_steps: number of MCMC steps
    burn_in: number of burn-in steps (discarded)
    step_size: step size for the update
    
    Returns:
    samples: array of samples, shape (n_steps - burn_in, n_visible)
    """
    current_state = initial_state.clone().requires_grad_(True)
    samples = []
    
    for step in range(n_steps):
        # Perform one Langevin dynamics step
        current_state = langevin_dynamics_step(wavefunction, current_state, step_size)
        current_state = current_state.detach().requires_grad_(True)
        
        # Store sample after burn-in
        if step >= burn_in:
            samples.append(current_state.clone())
    
    # Convert to tensor
    samples = torch.stack(samples, dim=0)
    
    return samples

# Example: Compare Metropolis-Hastings and Langevin dynamics
n_spins = 4
n_hidden = 4
rbm = SimpleRBM(n_spins, n_hidden).to(device)

# Sampling parameters
n_steps = 20000
burn_in = 2000
step_size = 0.1

# Initial state (all spins up)
initial_state = torch.ones(n_spins, device=device, requires_grad=True)

# Perform Metropolis-Hastings sampling
samples_mh, acceptance_rate_mh = metropolis_hastings_sampling(
    rbm, initial_state.detach(), n_steps, burn_in
)

# Perform Langevin dynamics sampling
samples_ld = langevin_dynamics_sampling(
    rbm, initial_state, n_steps, burn_in, step_size
)

print(f"Metropolis-Hastings:")
print(f"Number of samples: {len(samples_mh)}")
print(f"Acceptance rate: {acceptance_rate_mh:.4f}")

print(f"\nLangevin Dynamics:")
print(f"Number of samples: {len(samples_ld)}")

# Compute magnetization time series
magnetization_mh = torch.mean(samples_mh, dim=1)
magnetization_ld = torch.mean(samples_ld, dim=1)

# Compute autocorrelation functions
max_lag = 1000
autocorr_mh = compute_autocorrelation(magnetization_mh, max_lag)
autocorr_ld = compute_autocorrelation(magnetization_ld, max_lag)

# Plot autocorrelation functions
plt.figure(figsize=(12, 6))
plt.plot(autocorr_mh.cpu().numpy(), label='Metropolis-Hastings')
plt.plot(autocorr_ld.cpu().numpy(), label='Langevin Dynamics')
plt.xlabel('Lag')
plt.ylabel('Autocorrelation')
plt.title('Autocorrelation Function Comparison')
plt.legend()
plt.grid(True)
plt.show()

# Compute integrated autocorrelation times
tau_int_mh = compute_integrated_autocorrelation_time(autocorr_mh)
tau_int_ld = compute_integrated_autocorrelation_time(autocorr_ld)

print(f"\nMetropolis-Hastings:")
print(f"Integrated autocorrelation time: {tau_int_mh.item():.2f}")

print(f"\nLangevin Dynamics:")
print(f"Integrated autocorrelation time: {tau_int_ld.item():.2f}")

# Compute effective sample sizes
n_eff_mh = compute_effective_sample_size(magnetization_mh, autocorr_mh)
n_eff_ld = compute_effective_sample_size(magnetization_ld, autocorr_ld)

print(f"\nMetropolis-Hastings:")
print(f"Effective sample size: {n_eff_mh.item():.2f}")
print(f"Sampling efficiency: {n_eff_mh.item() / len(samples_mh) * 100:.2f}%")

print(f"\nLangevin Dynamics:")
print(f"Effective sample size: {n_eff_ld.item():.2f}")
print(f"Sampling efficiency: {n_eff_ld.item() / len(samples_ld) * 100:.2f}%")

# Plot magnetization time series
plt.figure(figsize=(12, 6))
plt.plot(magnetization_mh.cpu().numpy()[:1000], label='Metropolis-Hastings')
plt.plot(magnetization_ld.cpu().numpy()[:1000], label='Langevin Dynamics')
plt.xlabel('MCMC Step')
plt.ylabel('Magnetization')
plt.title('Magnetization Time Series Comparison (First 1000 steps)')
plt.legend()
plt.grid(True)
plt.show()

# Compare sampling methods for a larger system
n_spins_large = 8
n_hidden_large = 8
rbm_large = SimpleRBM(n_spins_large, n_hidden_large).to(device)

# Initial state (all spins up)
initial_state_large = torch.ones(n_spins_large, device=device, requires_grad=True)

# Perform sampling
n_steps_large = 10000
burn_in_large = 1000

print(f"\nSampling for larger system (n_spins = {n_spins_large}):")

# Metropolis-Hastings
samples_mh_large, acceptance_rate_mh_large = metropolis_hastings_sampling(
    rbm_large, initial_state_large.detach(), n_steps_large, burn_in_large
)
print(f"Metropolis-Hastings acceptance rate: {acceptance_rate_mh_large:.4f}")

# Langevin dynamics
samples_ld_large = langevin_dynamics_sampling(
    rbm_large, initial_state_large, n_steps_large, burn_in_large, step_size
)
print(f"Langevin dynamics completed")

# Compute magnetization time series
magnetization_mh_large = torch.mean(samples_mh_large, dim=1)
magnetization_ld_large = torch.mean(samples_ld_large, dim=1)

# Compute autocorrelation functions
max_lag_large = 500
autocorr_mh_large = compute_autocorrelation(magnetization_mh_large, max_lag_large)
autocorr_ld_large = compute_autocorrelation(magnetization_ld_large, max_lag_large)

# Compute integrated autocorrelation times
tau_int_mh_large = compute_integrated_autocorrelation_time(autocorr_mh_large)
tau_int_ld_large = compute_integrated_autocorrelation_time(autocorr_ld_large)

print(f"Metropolis-Hastings integrated autocorrelation time: {tau_int_mh_large.item():.2f}")
print(f"Langevin Dynamics integrated autocorrelation time: {tau_int_ld_large.item():.2f}")

# Compute effective sample sizes
n_eff_mh_large = compute_effective_sample_size(magnetization_mh_large, autocorr_mh_large)
n_eff_ld_large = compute_effective_sample_size(magnetization_ld_large, autocorr_ld_large)

print(f"Metropolis-Hastings effective sample size: {n_eff_mh_large.item():.2f}")
print(f"Metropolis-Hastings sampling efficiency: {n_eff_mh_large.item() / len(samples_mh_large) * 100:.2f}%")

print(f"Langevin Dynamics effective sample size: {n_eff_ld_large.item():.2f}")
print(f"Langevin Dynamics sampling efficiency: {n_eff_ld_large.item() / len(samples_ld_large) * 100:.2f}%")

## 总结

本节介绍了马尔可夫链蒙特卡洛采样方法，包括：

1. **马尔可夫链基础**：介绍了马尔可夫链的定义、转移概率、平稳分布和详细平衡条件

2. **Metropolis-Hastings算法**：详细介绍了Metropolis-Hastings算法的步骤、对称提议分布和自旋翻转提议

3. **退火方法**：介绍了模拟退火、退火调度和并行退火等提高采样效率的技术

4. **采样效率评估**：介绍了自相关时间、有效样本大小和接受率等评估采样效率的指标

5. **高级采样技术**：介绍了哈密顿蒙特卡洛、朗之文动力学和自适应提议分布等高级采样方法

这些采样方法是变分蒙特卡洛方法的核心技术，它们决定了从神经网络量子态分布中采样的效率和准确性。在实际应用中，我们需要根据具体问题的特点选择合适的采样方法。